# Advanced Model Comparison and Selection

**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**

## Learning Objectives

- Master systematic model comparison frameworks
- Use information criteria (AIC, BIC) effectively
- Implement nested and non-nested model tests
- Perform robust cross-validation strategies
- Build model ensembles for better predictions
- Understand the bias-variance tradeoff
- Select models based on business objectives


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)


## The Model Selection Problem

**Question:** How do we choose the best model?

**Considerations:**
1. **Predictive accuracy** - How well does it generalize?
2. **Interpretability** - Can we explain the predictions?
3. **Computational cost** - How fast is training/prediction?
4. **Robustness** - How sensitive to outliers/noise?
5. **Business constraints** - What matters to stakeholders?

**No free lunch:** No single model is best for all problems!


## Load Data

In [ ]:
# Generate comprehensive match dataset
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'shots': np.random.randint(5, 20, n),
    'shots_on_target': np.random.randint(2, 12, n),
    'possession': np.random.uniform(35, 70, n),
    'pass_accuracy': np.random.uniform(70, 92, n),
    'xg': np.random.uniform(0.5, 3.5, n),
    'opponent_xg': np.random.uniform(0.5, 3.5, n),
    'home': np.random.choice([0, 1], n),
    'opponent_strength': np.random.uniform(0.3, 0.9, n)
})

# Generate realistic goals
df['goals'] = (
    df['xg'] * 0.75 + 
    df['shots_on_target'] * 0.08 + 
    df['home'] * 0.3 - 
    df['opponent_strength'] * 0.4 +
    (df['possession'] - 50) * 0.01 +
    np.random.normal(0, 0.5, n)
).clip(0, 6).round().astype(int)

print(f"Dataset: {df.shape}")
print(df.head())

# Prepare features and target
feature_cols = ['shots', 'shots_on_target', 'possession', 'pass_accuracy', 
                'xg', 'opponent_xg', 'home', 'opponent_strength']
X = df[feature_cols]
y = df['goals']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"\
Train: {len(X_train)}, Test: {len(X_test)}")


## 1. Information Criteria: AIC and BIC

**AIC (Akaike Information Criterion):**
- Balances fit and complexity
- Penalizes number of parameters
- Lower is better

**BIC (Bayesian Information Criterion):**
- Similar to AIC but stronger penalty for complexity
- Prefers simpler models
- Lower is better

**When to use:**
- Comparing models with different numbers of features
- Non-nested models
- Want to avoid overfitting


In [ ]:
# Fit models with different feature sets using statsmodels
models_ic = {}

# Model 1: Basic features
formula1 = 'goals ~ xg + shots_on_target + home'
models_ic['Basic'] = smf.ols(formula1, data=pd.concat([X_train, y_train], axis=1)).fit()

# Model 2: Extended features
formula2 = 'goals ~ xg + shots_on_target + home + opponent_strength + possession'
models_ic['Extended'] = smf.ols(formula2, data=pd.concat([X_train, y_train], axis=1)).fit()

# Model 3: All features
formula3 = 'goals ~ ' + ' + '.join(feature_cols)
models_ic['Full'] = smf.ols(formula3, data=pd.concat([X_train, y_train], axis=1)).fit()

# Compare using AIC and BIC
comparison = []
for name, model in models_ic.items():
    comparison.append({
        'Model': name,
        'Features': model.df_model,
        'R²': model.rsquared,
        'Adj R²': model.rsquared_adj,
        'AIC': model.aic,
        'BIC': model.bic
    })

comparison_df = pd.DataFrame(comparison)
print("Information Criteria Comparison:")
print(comparison_df.to_string(index=False))

best_aic = comparison_df.loc[comparison_df['AIC'].idxmin(), 'Model']
best_bic = comparison_df.loc[comparison_df['BIC'].idxmin(), 'Model']

print(f"\
Best by AIC: {best_aic}")
print(f"Best by BIC: {best_bic}")
print(f"\
Note: BIC typically prefers simpler models than AIC.")


## 2. Cross-Validation Strategies

**Standard K-Fold:** Random splits
**Stratified K-Fold:** Preserves class distribution
**Time Series Split:** Respects temporal order
**Leave-One-Out:** Each observation is test set once


In [ ]:
# Compare different CV strategies
model = LinearRegression()

# 1. Standard K-Fold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(model, X, y, cv=kfold, scoring='r2')

# 2. Time Series Split (assuming data is ordered by time)
tscv = TimeSeriesSplit(n_splits=5)
scores_tscv = cross_val_score(model, X, y, cv=tscv, scoring='r2')

print("Cross-Validation Strategy Comparison:")
print(f"\
Standard K-Fold (5-fold):")
print(f"  Mean R²: {scores_kfold.mean():.3f} (±{scores_kfold.std():.3f})")
print(f"  Scores: {scores_kfold}")

print(f"\
Time Series Split (5-fold):")
print(f"  Mean R²: {scores_tscv.mean():.3f} (±{scores_tscv.std():.3f})")
print(f"  Scores: {scores_tscv}")

print(f"\
Note: Time Series Split is more realistic for temporal data.")
print(f"It trains on past data and tests on future data.")


## 3. Comprehensive Model Comparison

Let's compare multiple model types systematically.


In [ ]:
# Scale features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5),
    'KNN (K=7)': KNeighborsRegressor(n_neighbors=7),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
}

# Evaluate each model
results = []

for name, model in models.items():
    # Use scaled data for distance-based and regularized models
    use_scaled = name in ['Ridge (α=1.0)', 'Lasso (α=0.1)', 'ElasticNet', 'KNN (K=7)']
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    
    # Train
    model.fit(X_tr, y_train)
    
    # Predict
    y_train_pred = model.predict(X_tr)
    y_test_pred = model.predict(X_te)
    
    # Metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    # Cross-validation
    X_cv = scaler.fit_transform(X) if use_scaled else X
    cv_scores = cross_val_score(model, X_cv, y, cv=5, scoring='r2')
    
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'CV R² (mean)': cv_scores.mean(),
        'CV R² (std)': cv_scores.std(),
        'Test RMSE': test_rmse,
        'Test MAE': test_mae,
        'Overfit Gap': train_r2 - test_r2
    })

results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)

print("Comprehensive Model Comparison:")
print(results_df.to_string(index=False))

print(f"\
Best model by Test R²: {results_df.iloc[0]['Model']}")
print(f"Best model by CV R²: {results_df.loc[results_df['CV R² (mean)'].idxmax(), 'Model']}")
print(f"Least overfitting: {results_df.loc[results_df['Overfit Gap'].idxmin(), 'Model']}")


## Visualize Model Comparison

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Train vs Test R²
axes[0, 0].scatter(results_df['Train R²'], results_df['Test R²'], s=100, alpha=0.7)
for idx, row in results_df.iterrows():
    axes[0, 0].annotate(row['Model'], (row['Train R²'], row['Test R²']), 
                       fontsize=8, ha='right')
axes[0, 0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Generalization')
axes[0, 0].set_xlabel('Train R²')
axes[0, 0].set_ylabel('Test R²')
axes[0, 0].set_title('Train vs Test Performance')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Test R² with error bars (CV std)
axes[0, 1].barh(results_df['Model'], results_df['Test R²'], alpha=0.7)
axes[0, 1].errorbar(results_df['Test R²'], range(len(results_df)), 
                    xerr=results_df['CV R² (std)'], fmt='none', color='red', capsize=5)
axes[0, 1].set_xlabel('Test R²')
axes[0, 1].set_title('Test R² with CV Uncertainty')
axes[0, 1].grid(True, alpha=0.3, axis='x')

# 3. RMSE comparison
axes[1, 0].bar(range(len(results_df)), results_df['Test RMSE'], alpha=0.7, color='coral')
axes[1, 0].set_xticks(range(len(results_df)))
axes[1, 0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[1, 0].set_ylabel('Test RMSE')
axes[1, 0].set_title('Test RMSE (Lower is Better)')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Overfitting gap
colors = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' for x in results_df['Overfit Gap']]
axes[1, 1].barh(results_df['Model'], results_df['Overfit Gap'], color=colors, alpha=0.7)
axes[1, 1].axvline(x=0.05, color='orange', linestyle='--', label='Acceptable (<0.05)')
axes[1, 1].axvline(x=0.1, color='red', linestyle='--', label='Concerning (>0.1)')
axes[1, 1].set_xlabel('Overfit Gap (Train R² - Test R²)')
axes[1, 1].set_title('Overfitting Analysis')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Top-left: Points near diagonal = good generalization")
print("- Top-right: Longer error bars = higher variance")
print("- Bottom-left: Lower bars = better predictions")
print("- Bottom-right: Green = minimal overfitting, Red = severe overfitting")


## 4. Model Ensembles

**Idea:** Combine multiple models for better predictions.

**Methods:**
1. **Simple averaging:** Average predictions from all models
2. **Weighted averaging:** Weight by performance
3. **Stacking:** Train a meta-model on predictions


In [ ]:
# Create ensemble predictions
ensemble_models = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
}

# Train models and get predictions
predictions = {}
for name, model in ensemble_models.items():
    use_scaled = name == 'Ridge'
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    
    model.fit(X_tr, y_train)
    predictions[name] = model.predict(X_te)

# 1. Simple averaging
ensemble_simple = np.mean(list(predictions.values()), axis=0)

# 2. Weighted averaging (by test R²)
weights = []
for name in ensemble_models.keys():
    pred = predictions[name]
    r2 = r2_score(y_test, pred)
    weights.append(r2)

weights = np.array(weights) / np.sum(weights)  # Normalize
ensemble_weighted = np.average(list(predictions.values()), axis=0, weights=weights)

# Evaluate ensembles
print("Ensemble Performance:")
print(f"\
Individual Models:")
for name, pred in predictions.items():
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(f"  {name}: R² = {r2:.3f}, RMSE = {rmse:.3f}")

print(f"\
Ensembles:")
r2_simple = r2_score(y_test, ensemble_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, ensemble_simple))
print(f"  Simple Average: R² = {r2_simple:.3f}, RMSE = {rmse_simple:.3f}")

r2_weighted = r2_score(y_test, ensemble_weighted)
rmse_weighted = np.sqrt(mean_squared_error(y_test, ensemble_weighted))
print(f"  Weighted Average: R² = {r2_weighted:.3f}, RMSE = {rmse_weighted:.3f}")

if r2_weighted > max([r2_score(y_test, p) for p in predictions.values()]):
    print(f"\
✅ Ensemble outperforms individual models!")
else:
    print(f"\
⚠️ Best individual model still performs better.")


## 5. Bias-Variance Tradeoff

**Bias:** Error from wrong assumptions (underfitting)
**Variance:** Error from sensitivity to training data (overfitting)

**Goal:** Find the sweet spot!


In [ ]:
# Visualize bias-variance tradeoff with polynomial features
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Use single feature for visualization
X_simple = df[['xg']].values
y_simple = df['goals'].values

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.25, random_state=42
)

# Test different polynomial degrees
degrees = range(1, 11)
train_scores = []
test_scores = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train_s, y_train_s)
    
    train_scores.append(model.score(X_train_s, y_train_s))
    test_scores.append(model.score(X_test_s, y_test_s))

# Plot
plt.figure(figsize=(12, 6))
plt.plot(degrees, train_scores, 'o-', label='Training R²', linewidth=2)
plt.plot(degrees, test_scores, 's-', label='Test R²', linewidth=2)
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('R² Score')
plt.title('Bias-Variance Tradeoff')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axvline(x=degrees[np.argmax(test_scores)], color='green', linestyle='--', 
            label=f'Optimal Complexity (degree={degrees[np.argmax(test_scores)]})')
plt.legend()
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Low degree: High bias (underfitting) - both scores low")
print("- Optimal degree: Best test performance")
print("- High degree: High variance (overfitting) - train high, test low")
print(f"\
Optimal polynomial degree: {degrees[np.argmax(test_scores)]}")


## 6. Model Selection Framework

A systematic approach to choosing models.


In [ ]:
def model_selection_framework(results_df, priorities):
    """
    Select best model based on priorities.
    
    priorities: dict with keys 'accuracy', 'interpretability', 'speed', 'robustness'
    Each value is a weight from 0 to 1 (should sum to 1)
    """
    
    # Define model characteristics (0-1 scale)
    characteristics = {
        'Linear Regression': {'accuracy': 0.7, 'interpretability': 1.0, 'speed': 1.0, 'robustness': 0.6},
        'Ridge (α=1.0)': {'accuracy': 0.75, 'interpretability': 0.9, 'speed': 0.9, 'robustness': 0.7},
        'Lasso (α=0.1)': {'accuracy': 0.75, 'interpretability': 0.95, 'speed': 0.9, 'robustness': 0.7},
        'ElasticNet': {'accuracy': 0.75, 'interpretability': 0.9, 'speed': 0.9, 'robustness': 0.75},
        'KNN (K=7)': {'accuracy': 0.8, 'interpretability': 0.4, 'speed': 0.5, 'robustness': 0.5},
        'Decision Tree': {'accuracy': 0.75, 'interpretability': 0.8, 'speed': 0.8, 'robustness': 0.4},
        'Random Forest': {'accuracy': 0.85, 'interpretability': 0.5, 'speed': 0.6, 'robustness': 0.8},
        'Gradient Boosting': {'accuracy': 0.9, 'interpretability': 0.4, 'speed': 0.5, 'robustness': 0.7}
    }
    
    # Normalize accuracy by test R²
    max_r2 = results_df['Test R²'].max()
    for model_name in characteristics.keys():
        if model_name in results_df['Model'].values:
            actual_r2 = results_df[results_df['Model'] == model_name]['Test R²'].values[0]
            characteristics[model_name]['accuracy'] = actual_r2 / max_r2
    
    # Calculate weighted scores
    scores = {}
    for model_name, chars in characteristics.items():
        score = sum(chars[key] * priorities[key] for key in priorities.keys())
        scores[model_name] = score
    
    # Rank models
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    return ranked

# Example 1: Prioritize accuracy
print("Scenario 1: Research Project (Accuracy Priority)")
priorities_accuracy = {
    'accuracy': 0.7,
    'interpretability': 0.1,
    'speed': 0.1,
    'robustness': 0.1
}
ranked_accuracy = model_selection_framework(results_df, priorities_accuracy)
print("Top 3 models:")
for i, (model, score) in enumerate(ranked_accuracy[:3], 1):
    print(f"  {i}. {model}: {score:.3f}")

# Example 2: Prioritize interpretability
print("\
Scenario 2: Stakeholder Presentation (Interpretability Priority)")
priorities_interpret = {
    'accuracy': 0.3,
    'interpretability': 0.5,
    'speed': 0.1,
    'robustness': 0.1
}
ranked_interpret = model_selection_framework(results_df, priorities_interpret)
print("Top 3 models:")
for i, (model, score) in enumerate(ranked_interpret[:3], 1):
    print(f"  {i}. {model}: {score:.3f}")

# Example 3: Balanced
print("\
Scenario 3: Production System (Balanced)")
priorities_balanced = {
    'accuracy': 0.4,
    'interpretability': 0.2,
    'speed': 0.2,
    'robustness': 0.2
}
ranked_balanced = model_selection_framework(results_df, priorities_balanced)
print("Top 3 models:")
for i, (model, score) in enumerate(ranked_balanced[:3], 1):
    print(f"  {i}. {model}: {score:.3f}")


## 7. Statistical Model Comparison Tests

**For nested models:** Use F-test or likelihood ratio test
**For non-nested models:** Use AIC/BIC or cross-validation


In [ ]:
# Nested model comparison using F-test
train_data = pd.concat([X_train, y_train], axis=1)

# Reduced model
model_reduced = smf.ols('goals ~ xg + home', data=train_data).fit()

# Full model
model_full = smf.ols('goals ~ ' + ' + '.join(feature_cols), data=train_data).fit()

# F-test
f_test = model_reduced.compare_f_test(model_full)

print("Nested Model Comparison (F-test):")
print(f"\
Reduced Model: goals ~ xg + home")
print(f"  R²: {model_reduced.rsquared:.3f}")
print(f"  Parameters: {model_reduced.df_model}")

print(f"\
Full Model: goals ~ all features")
print(f"  R²: {model_full.rsquared:.3f}")
print(f"  Parameters: {model_full.df_model}")

print(f"\
F-statistic: {f_test[0][0]:.3f}")
print(f"p-value: {f_test[1]:.4f}")

if f_test[1] < 0.05:
    print("\
✅ Full model is significantly better than reduced model.")
else:
    print("\
⚠️ Additional features do not significantly improve the model.")
    print("Consider using the simpler model (Occam's Razor).")


## Summary

In this notebook, we:

1. ✅ Used information criteria (AIC, BIC) for model selection
2. ✅ Implemented multiple cross-validation strategies
3. ✅ Compared 8 different regression models systematically
4. ✅ Built model ensembles for improved predictions
5. ✅ Visualized the bias-variance tradeoff
6. ✅ Created a model selection framework based on priorities
7. ✅ Performed statistical tests for nested models

## Key Takeaways

- **No single best model** - depends on context and priorities
- **AIC/BIC** help compare models with different complexity
- **Cross-validation** provides robust performance estimates
- **Ensembles** often outperform individual models
- **Bias-variance tradeoff** guides model complexity
- **Business objectives** should drive model selection
- **Statistical tests** formalize model comparisons
- **Interpretability** matters for stakeholder buy-in

## Model Selection Checklist

✅ Define success criteria (accuracy, interpretability, speed)
✅ Split data properly (train/validation/test)
✅ Compare multiple model types
✅ Use cross-validation for robust estimates
✅ Check for overfitting (train vs test gap)
✅ Consider ensemble methods
✅ Validate on holdout test set
✅ Document model limitations
✅ Monitor performance in production
✅ Update models regularly with new data


## Practice Exercises

1. **Nested Cross-Validation**: Implement nested CV for hyperparameter tuning
2. **Bayesian Model Selection**: Use Bayesian methods for model comparison
3. **Online Learning**: Implement incremental model updates
4. **A/B Testing**: Design an A/B test framework for model comparison in production
5. **Cost-Sensitive Learning**: Incorporate prediction costs into model selection
6. **Multi-Objective Optimization**: Balance multiple objectives (accuracy, fairness, speed)
7. **AutoML**: Explore automated model selection tools
